In [1]:
import random

import carla


CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT_SECONDS = 10.0

NUMBER_OF_PEDESTRIANS = 10
RANDOM_SEED = 42

In [ ]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:
    """Create a CARLA client and return the connected world."""
    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT_SECONDS)

    return client, client.get_world()


In [3]:
def get_pedestrian_blueprints(
    blueprint_library: carla.BlueprintLibrary,
) -> list[carla.ActorBlueprint]:
    """Return all available pedestrian blueprints."""
    blueprints = list(
        blueprint_library.filter("walker.pedestrian.*")
    )

    if not blueprints:
        raise RuntimeError("No pedestrian blueprints are available.")

    return blueprints

In [4]:
def get_spawn_location(world: carla.World) -> carla.Transform | None:
    """Return a random valid pedestrian navigation location."""
    location = world.get_random_location_from_navigation()

    if location is None:
        return None

    return carla.Transform(location)


In [5]:
def spawn_pedestrians(
    world: carla.World,
    pedestrian_blueprints: list[carla.ActorBlueprint],
    count: int,
) -> list[carla.Walker]:
    """Spawn multiple pedestrians and return successfully created actors."""
    pedestrians: list[carla.Walker] = []

    for index in range(count):
        spawn_transform = get_spawn_location(world)

        if spawn_transform is None:
            print(
                f"[WARNING] No valid navigation location found "
                f"for pedestrian {index + 1}."
            )
            continue

        pedestrian_bp = random.choice(pedestrian_blueprints)

        pedestrian = world.try_spawn_actor(
            pedestrian_bp,
            spawn_transform,
        )

        if pedestrian is None:
            print(
                f"[WARNING] Failed to spawn pedestrian "
                f"{index + 1}."
            )
            continue

        pedestrians.append(pedestrian)

        location = pedestrian.get_location()

        print(
            f"[{len(pedestrians):02d}] "
            f"ID={pedestrian.id} "
            f"Type={pedestrian.type_id} "
            f"Location=("
            f"{location.x:.2f}, "
            f"{location.y:.2f}, "
            f"{location.z:.2f})"
        )

    return pedestrians

In [6]:
def main() -> None:
    """Spawn and report multiple pedestrians."""
    random.seed(RANDOM_SEED)

    client, world = connect_to_carla()
    blueprint_library = world.get_blueprint_library()

    pedestrian_blueprints = get_pedestrian_blueprints(
        blueprint_library
    )

    print("=== SPAWNING PEDESTRIANS ===")
    print(
        f"Available pedestrian blueprints: "
        f"{len(pedestrian_blueprints)}"
    )
    print(
        f"Requested pedestrians: "
        f"{NUMBER_OF_PEDESTRIANS}"
    )
    print()

    pedestrians = spawn_pedestrians(
        world,
        pedestrian_blueprints,
        NUMBER_OF_PEDESTRIANS,
    )

    print()
    print("=== SPAWN SUMMARY ===")
    print(
        f"Successfully spawned: "
        f"{len(pedestrians)}/{NUMBER_OF_PEDESTRIANS}"
    )

In [7]:
if __name__ == "__main__":
    main()

=== SPAWNING PEDESTRIANS ===
Available pedestrian blueprints: 52
Requested pedestrians: 10

[WARNING] Failed to spawn pedestrian 1.
[01] ID=25 Type=walker.pedestrian.0013 Location=(0.00, 0.00, 0.00)
[02] ID=26 Type=walker.pedestrian.0028 Location=(0.00, 0.00, 0.00)
[03] ID=27 Type=walker.pedestrian.0048 Location=(0.00, 0.00, 0.00)
[04] ID=28 Type=walker.pedestrian.0006 Location=(0.00, 0.00, 0.00)
[05] ID=29 Type=walker.pedestrian.0004 Location=(0.00, 0.00, 0.00)
[06] ID=30 Type=walker.pedestrian.0003 Location=(0.00, 0.00, 0.00)
[07] ID=31 Type=walker.pedestrian.0031 Location=(0.00, 0.00, 0.00)
[08] ID=32 Type=walker.pedestrian.0048 Location=(0.00, 0.00, 0.00)
[09] ID=33 Type=walker.pedestrian.0021 Location=(0.00, 0.00, 0.00)

=== SPAWN SUMMARY ===
Successfully spawned: 9/10
